In [0]:
"""
Data Quality - Generate Dataset Summary Task
Aggregates column summaries to dataset-level metrics
"""
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import json

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("enabled_dimensions", "[]")
dbutils.widgets.text("threshold_ok", "100.0")
dbutils.widgets.text("threshold_warning", "98.0")
dbutils.widgets.text("thresholds_json", "{}")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
ENABLED_DIMENSIONS = json.loads(dbutils.widgets.get("enabled_dimensions"))
THRESHOLD_OK = float(dbutils.widgets.get("threshold_ok"))
THRESHOLD_WARNING = float(dbutils.widgets.get("threshold_warning"))
THRESHOLDS_CONFIG = json.loads(dbutils.widgets.get("thresholds_json") or "{}")

# Sanitize names for table naming
import re
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Generate Dataset Summary ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Results Schema: {RESULTS_SCHEMA}")
print(f"Enabled dimensions: {ENABLED_DIMENSIONS}")
print(f"Thresholds: OK ≥ {THRESHOLD_OK}%, Warning ≥ {THRESHOLD_WARNING}%")

# Read column summary
column_summary_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_summary_columns"
print(f"\nReading from: {column_summary_table}")

try:
    df_column_summary = spark.table(column_summary_table)
    row_count = df_column_summary.count()
    print(f"✓ Found {row_count} column summaries")
    
    if row_count > 0:
        print(f"Available columns: {df_column_summary.columns}")
        print(f"Sample data:")
        df_column_summary.show(5, truncate=False)
except Exception as e:
    print(f"✗ Error reading column summary table: {str(e)}")
    raise

# Define output schema
dataset_schema = StructType([
    StructField("check_type", StringType(), False),
    StructField("dataset_score", DoubleType(), True),
    StructField("dataset_status", StringType(), True),
    StructField("total_columns", IntegerType(), False),
    StructField("columns_ok", IntegerType(), False),
    StructField("columns_warning", IntegerType(), False),
    StructField("columns_ko", IntegerType(), False),
    StructField("execution_timestamp", StringType(), False)
])

summary_data = []
execution_timestamp = spark.sql("SELECT current_timestamp() as ts").collect()[0]['ts']

def determine_status(score):
    """Determine status based on score"""
    if score is None:
        return None
    if score >= THRESHOLD_OK:
        return "OK"
    elif score >= THRESHOLD_WARNING:
        return "Warning"
    else:
        return "KO"

def dimension_thresholds(dimension, summary_frame):
    configured = THRESHOLDS_CONFIG.get("by_dimension", {}).get(dimension, {})
    values = [configured.get(row["column_name"], {}) for row in summary_frame.select("column_name").collect()]
    ok_values = [float(value["ok"]) for value in values if "ok" in value]
    warning_values = [float(value["warning"]) for value in values if "warning" in value]
    return (
        sum(ok_values) / len(ok_values) if ok_values else THRESHOLD_OK,
        sum(warning_values) / len(warning_values) if warning_values else THRESHOLD_WARNING
    )

print("\nDataset-level aggregation:")

# Check if column summary table has data
if df_column_summary.count() == 0:
    print("WARNING: Column summary table is empty!")
    print("Creating empty dataset summary table...")
    
    # Create empty dataset summary with schema
    df_dataset_summary = spark.createDataFrame([], dataset_schema)
else:
    # Aggregate for each dimension + overall
    for dimension in ENABLED_DIMENSIONS + ['overall']:
        print(f"\nProcessing dimension: {dimension}")
        
        # Filter for this dimension
        df_dim = df_column_summary.filter(F.col("check_type") == dimension)
        dim_count = df_dim.count()
        
        print(f"  Found {dim_count} column summaries for {dimension}")
        
        if dim_count == 0:
            print(f"  ⊘ No data for {dimension}, skipping")
            continue
        
        # Calculate dataset score (average of column scores)
        dataset_score = df_dim.select(F.avg("score")).collect()[0][0]
        dataset_score = round(dataset_score, 2) if dataset_score is not None else None

        dimension_ok, dimension_warning = dimension_thresholds(dimension, df_dim)
        
        # Count columns by status
        status_counts = df_dim.groupBy("status").count().collect()
        status_dict = {row['status']: row['count'] for row in status_counts}
        
        columns_ok = status_dict.get("OK", 0)
        columns_warning = status_dict.get("Warning", 0)
        columns_ko = status_dict.get("KO", 0)
        total_columns = columns_ok + columns_warning + columns_ko
        
        if dataset_score is None:
            dataset_status = None
        elif dataset_score >= dimension_ok:
            dataset_status = "OK"
        elif dataset_score >= dimension_warning:
            dataset_status = "Warning"
        else:
            dataset_status = "KO"
        
        print(f"  ✓ {dimension}: {dataset_score}% ({dataset_status})")
        print(f"    - OK: {columns_ok}, Warning: {columns_warning}, KO: {columns_ko}, Total: {total_columns}")
        
        summary_data.append((
            dimension,
            dataset_score,
            dataset_status,
            total_columns,
            columns_ok,
            columns_warning,
            columns_ko,
            str(execution_timestamp)
        ))
    
    # Create dataset summary DataFrame
    df_dataset_summary = spark.createDataFrame(summary_data, dataset_schema)

print(f"\nTotal dataset summary rows created: {len(summary_data)}")

# Output table name
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_summary_dataset"

print(f"\nWriting to: {output_table}")

# Write to Unity Catalog
df_dataset_summary.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_dataset_summary.count()} rows to {output_table}")

# Display summary
print("\nDataset Summary:")
df_dataset_summary.show(truncate=False)